# Impulse Ramp + Butterworth Zero-Phase Filter
Set the parameters in the next cell, run it, and the plot will show a raw impulse-like ramp and its zero-phase filtered version with smoothed transitions.

In [27]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from scipy.signal import butter, filtfilt

In [28]:
def make_impulse_ramp(
    n_points,
    init_rm_pct,
    final_rm_pct,
    ramp_slope,
    start_amp,
    end_amp,
):
    """
    Build an impulse-like ramp with configurable amplitudes.
    - start region removed (forced to start_amp)
    - middle region rises/falls toward end_amp according to slope
    - end region removed (forced to end_amp)
    """
    init_rm = int(np.clip(init_rm_pct, 0, 100) / 100.0 * n_points)
    final_rm = int(np.clip(final_rm_pct, 0, 100) / 100.0 * n_points)

    if init_rm + final_rm >= n_points:
        raise ValueError("Initial + final removed points exceed signal size.")

    raw = np.full(n_points, start_amp, dtype=float)
    active_start = init_rm
    active_end = n_points - final_rm
    active_len = active_end - active_start

    if active_len <= 0:
        return raw

    direction = np.sign(end_amp - start_amp) if end_amp != start_amp else 1.0
    idx = np.arange(active_len, dtype=float)
    active_ramp = start_amp + direction * ramp_slope * idx

    lo = min(start_amp, end_amp)
    hi = max(start_amp, end_amp)
    active_ramp = np.clip(active_ramp, lo, hi)

    raw[active_start:active_end] = active_ramp
    if final_rm > 0:
        raw[-final_rm:] = end_amp

    return raw


def butterworth_iir_coeffs(filter_type, fs, filt_order, cutoff_hz=None, lowcut_hz=None, highcut_hz=None):
    """
    Return Butterworth IIR coefficients (b, a) for an explicit filter type.

    Parameters
    ----------
    filter_type : str
        One of: "lowpass", "highpass", "bandpass"
    fs : float
        Sampling frequency in Hz
    filt_order : int
        Butterworth filter order
    cutoff_hz : float, optional
        Cutoff frequency used for low-pass or high-pass
    lowcut_hz : float, optional
        Lower band edge for band-pass
    highcut_hz : float, optional
        Upper band edge for band-pass
    """
    if fs <= 0:
        raise ValueError("Sampling frequency must be > 0.")

    nyq = 0.5 * fs
    filter_type = filter_type.lower().strip()

    if filter_type == "lowpass":
        if cutoff_hz is None or not (0 < cutoff_hz < nyq):
            raise ValueError("For lowpass, cutoff_hz must satisfy 0 < cutoff_hz < Nyquist.")
        wn = cutoff_hz / nyq
        b, a = butter(filt_order, wn, btype="low")
        mode = "low-pass"
    elif filter_type == "highpass":
        if cutoff_hz is None or not (0 < cutoff_hz < nyq):
            raise ValueError("For highpass, cutoff_hz must satisfy 0 < cutoff_hz < Nyquist.")
        wn = cutoff_hz / nyq
        b, a = butter(filt_order, wn, btype="high")
        mode = "high-pass"
    elif filter_type == "bandpass":
        if lowcut_hz is None or highcut_hz is None:
            raise ValueError("For bandpass, both lowcut_hz and highcut_hz are required.")
        if not (0 < lowcut_hz < highcut_hz < nyq):
            raise ValueError("For bandpass, require 0 < lowcut_hz < highcut_hz < Nyquist.")
        wn = [lowcut_hz / nyq, highcut_hz / nyq]
        b, a = butter(filt_order, wn, btype="band")
        mode = "band-pass"
    else:
        raise ValueError("filter_type must be one of: 'lowpass', 'highpass', 'bandpass'.")

    return b, a, mode

In [29]:


# -----------------------------
# 1) User-defined ramp parameters
# -----------------------------
size = 40961                  # Number of points in the signal
initial_remove_pct = 10.0     # % of points removed at the beginning
final_remove_pct = 10.0       # % of points removed at the end
slope = 0.1                   # Ramp slope (larger -> faster rise)
init_value = 0.0              # Ramp starting amplitude
final_value = 1.0             # Ramp ending amplitude

# ----------------------------------------
# 2) User-defined Butterworth filter setup
# ----------------------------------------
filter_type = "lowpass"       # Options: "lowpass", "highpass", "bandpass"
sampling_freq_hz = 500e3      # Sampling frequency in Hz
order = 2                     # Butterworth filter order

# Use cutoff_hz for lowpass/highpass.
cutoff_hz = 150

# Use lowcut_hz and highcut_hz only for bandpass.
lowcut_hz = 1e3
highcut_hz = 10e3

# ----------------------------------------
# 3) Plot export options
# ----------------------------------------
save_as_svg = True
svg_filepath = "/Users/evillz/Downloads/impulse_ramp_zero_phase.svg"
svg_x_range = [0.0, 0.02]
svg_y_range = [-0.1, 1.1]

# Build raw signal
raw_ramp = make_impulse_ramp(
    size,
    initial_remove_pct,
    final_remove_pct,
    slope,
    init_value,
    final_value,
)
time = np.arange(size) / sampling_freq_hz

# 4) Design Butterworth IIR and apply zero-phase filter
b, a, filter_mode = butterworth_iir_coeffs(
    filter_type=filter_type,
    fs=sampling_freq_hz,
    filt_order=order,
    cutoff_hz=cutoff_hz,
    lowcut_hz=lowcut_hz,
    highcut_hz=highcut_hz,
)
filtered_ramp = filtfilt(b, a, raw_ramp)

In [30]:
print(f"Filter mode: {filter_mode}")
print("b coefficients:", b)
print("a coefficients:", a)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=time,
        y=raw_ramp,
        mode="lines",
        name="Ramp",
        line=dict(width=2),
        opacity=0.75,
    )
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=filtered_ramp,
        mode="lines",
        name="Filtered",
        line=dict(width=3),
    )
)
fig.update_layout(
    title="Impulse-Ramp Through Butterworth Zero-Phase Filter",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude",
    template="plotly_white",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=85, r=30, t=60, b=60),
)
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor="black",
    mirror=False,
    ticks="outside",
    automargin=True,
)
fig.update_yaxes(
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor="black",
    mirror=False,
    ticks="outside",
    automargin=True,
)

fig.show()

if save_as_svg:
    try:
        output_path = Path(svg_filepath).expanduser()
        output_path.parent.mkdir(parents=True, exist_ok=True)

        export_fig = go.Figure(fig)
        if svg_x_range is not None:
            export_fig.update_xaxes(range=svg_x_range)
        if svg_y_range is not None:
            export_fig.update_yaxes(range=svg_y_range)

        export_fig.update_layout(margin=dict(l=85, r=30, t=60, b=60))
        export_fig.update_xaxes(showgrid=False, zeroline=False, showline=True, linecolor="black", mirror=False, ticks="outside", automargin=True)
        export_fig.update_yaxes(showgrid=False, zeroline=False, showline=True, linecolor="black", mirror=False, ticks="outside", automargin=True)

        export_fig.write_image(str(output_path))
        print(f"Saved SVG to {output_path.resolve()}")
    except ValueError:
        print("SVG export requires the kaleido package. Install it with: pip install kaleido")

Filter mode: low-pass
b coefficients: [8.87081774e-07 1.77416355e-06 8.87081774e-07]
a coefficients: [ 1.         -1.99733427  0.99733782]


Saved SVG to /Users/evillz/Downloads/impulse_ramp_zero_phase.svg


In [ ]:
def make_ramp_function(
    n_points,
    init_rm_pct,
    final_rm_pct,
    ramp_slope,
    start_amp,
    end_amp,
):
    """
    Build a ramp-like signal with flat start/end regions and a linear ramp in between.
    This is similar to the impulse-ramp version, but the active section is treated as a ramp.
    """
    init_rm = int(np.clip(init_rm_pct, 0, 100) / 100.0 * n_points)
    final_rm = int(np.clip(final_rm_pct, 0, 100) / 100.0 * n_points)

    if init_rm + final_rm >= n_points:
        raise ValueError("Initial + final removed points exceed signal size.")

    raw = np.full(n_points, start_amp, dtype=float)
    active_start = init_rm
    active_end = n_points - final_rm
    active_len = active_end - active_start

    if active_len <= 0:
        return raw

    idx = np.arange(active_len, dtype=float)
    active_ramp = start_amp + ramp_slope * idx
    lo = min(start_amp, end_amp)
    hi = max(start_amp, end_amp)
    active_ramp = np.clip(active_ramp, lo, hi)

    raw[active_start:active_end] = active_ramp
    if final_rm > 0:
        raw[-final_rm:] = end_amp

    return raw


# Build ramp signal with the same parameters used above
raw_ramp_function = make_ramp_function(
    size,
    initial_remove_pct,
    final_remove_pct,
    slope,
    init_value,
    final_value,
)
filtered_ramp_function = filtfilt(b, a, raw_ramp_function)

fig_ramp = go.Figure()
fig_ramp.add_trace(
    go.Scatter(
        x=time,
        y=raw_ramp_function,
        mode="lines",
        name="Ramp function",
        line=dict(width=2),
        opacity=0.75,
    )
)
fig_ramp.add_trace(
    go.Scatter(
        x=time,
        y=filtered_ramp_function,
        mode="lines",
        name="Zero-phase Butterworth filtered",
        line=dict(width=3),
    )
)
fig_ramp.update_layout(
    title="Ramp Function Through Butterworth Zero-Phase Filter",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude",
    template="plotly_white",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=85, r=30, t=60, b=60),
)
fig_ramp.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor="black",
    mirror=False,
    ticks="outside",
    automargin=True,
)
fig_ramp.update_yaxes(
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor="black",
    mirror=False,
    ticks="outside",
    automargin=True,
)

fig_ramp.show()

if save_as_svg:
    try:
        output_path_ramp = Path(svg_filepath).expanduser().with_name(Path(svg_filepath).stem + "_ramp.svg")
        output_path_ramp.parent.mkdir(parents=True, exist_ok=True)

        export_fig_ramp = go.Figure(fig_ramp)
        if svg_x_range is not None:
            export_fig_ramp.update_xaxes(range=svg_x_range)
        if svg_y_range is not None:
            export_fig_ramp.update_yaxes(range=svg_y_range)

        export_fig_ramp.update_layout(margin=dict(l=85, r=30, t=60, b=60))
        export_fig_ramp.update_xaxes(showgrid=False, zeroline=False, showline=True, linecolor="black", mirror=False, ticks="outside", automargin=True)
        export_fig_ramp.update_yaxes(showgrid=False, zeroline=False, showline=True, linecolor="black", mirror=False, ticks="outside", automargin=True)

        export_fig_ramp.write_image(str(output_path_ramp))
        print(f"Saved SVG to {output_path_ramp.resolve()}")
    except ValueError:
        print("SVG export requires the kaleido package. Install it with: pip install kaleido")